# News Topic Classifier Using BERT



In [1]:
# Run this first in Colab
!pip install transformers datasets evaluate gradio accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

# Load AG News - it has 'train' and 'test' splits
dataset = load_dataset("ag_news")

# Quick look at what we're working with
print(dataset)
print("\nExample entry:")
print(dataset["train"][0])

# Label names: 0=World, 1=Sports, 2=Business, 3=Sci/Tech
label_names = dataset["train"].features["label"].names
print("\nLabel categories:", label_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Example entry:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Label categories: ['World', 'Sports', 'Business', 'Sci/Tech']


In [3]:
from transformers import AutoTokenizer

# Load BERT's tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    # Truncate/pad all texts to 128 tokens (headlines are short, 128 is enough)
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization to the full dataset
tokenized_dataset = dataset.map(tokenize, batched=True)

# Remove raw text column - model only needs token IDs
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Rename label → labels (what Hugging Face Trainer expects)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Set format to PyTorch tensors
tokenized_dataset.set_format("torch")

print("Tokenization done!")
print("Columns:", tokenized_dataset["train"].column_names)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Tokenization done!
Columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [4]:
from transformers import AutoModelForSequenceClassification

# Load BERT with a classification head (4 output classes)
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

# Map label IDs to human-readable names (for the Gradio demo later)
model.config.id2label = {i: name for i, name in enumerate(label_names)}
model.config.label2id = {name: i for i, name in enumerate(label_names)}

print("Model loaded!")
print("Output labels:", model.config.id2label)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!
Output labels: {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}


In [5]:
import evaluate
import numpy as np

# Load accuracy and F1 metric
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert raw scores to predicted class (highest score wins)
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",        # ✅ renamed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)

In [7]:
from transformers import Trainer

# Use a small subset for faster training in Colab (remove slicing for full training)
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(20000))
eval_dataset  = tokenized_dataset["test"].select(range(2000))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
print("Training complete!")

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.271956,0.261916,0.916000,0.915896
2,0.186782,0.231210,0.930000,0.929754
3,0.115393,0.236085,0.936000,0.935807


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training complete!


In [8]:
results = trainer.evaluate(tokenized_dataset["test"])

print("\n📊 Final Evaluation Results:")
print(f"  Accuracy : {results['eval_accuracy']:.4f}  ({results['eval_accuracy']*100:.2f}%)")
print(f"  F1 Score : {results['eval_f1']:.4f}")


📊 Final Evaluation Results:
  Accuracy : 0.9271  (92.71%)
  F1 Score : 0.9270


In [9]:
model.save_pretrained("./bert-ag-news-final")
tokenizer.save_pretrained("./bert-ag-news-final")
print("Model saved to ./bert-ag-news-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bert-ag-news-final


In [10]:
!zip -r bert-ag-news-final.zip bert-ag-news-final

  adding: bert-ag-news-final/ (stored 0%)
  adding: bert-ag-news-final/model.safetensors (deflated 7%)
  adding: bert-ag-news-final/config.json (deflated 54%)
  adding: bert-ag-news-final/tokenizer_config.json (deflated 42%)
  adding: bert-ag-news-final/tokenizer.json (deflated 71%)


In [11]:
from google.colab import files
files.download("bert-ag-news-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import gradio as gr
import torch
from transformers import pipeline

# Create a simple classification pipeline
classifier = pipeline(
    "text-classification",
    model="./bert-ag-news-final",
    tokenizer="./bert-ag-news-final",
    device=0 if torch.cuda.is_available() else -1
)

def classify_headline(text):
    if not text.strip():
        return "Please enter a news headline."

    result = classifier(text)[0]
    label = result["label"]
    score = result["score"]

    # Format all label scores for a nice output
    all_results = classifier(text, top_k=4)
    output = f"🏆 Predicted: **{label}** ({score*100:.1f}% confident)\n\n"
    output += "All scores:\n"
    for r in all_results:
        bar = "█" * int(r["score"] * 20)
        output += f"  {r['label']:10s} {bar} {r['score']*100:.1f}%\n"

    return output

# Build the UI
demo = gr.Interface(
    fn=classify_headline,
    inputs=gr.Textbox(
        placeholder="Paste a news headline here...",
        label="News Headline"
    ),
    outputs=gr.Textbox(label="Classification Result"),
    title="📰 News Topic Classifier",
    description="Fine-tuned BERT classifies headlines into: World | Sports | Business | Sci/Tech",
    examples=[
        ["NASA launches new telescope to explore deep space"],
        ["Stock markets fall amid inflation fears"],
        ["Manchester United wins the Premier League title"],
        ["World leaders meet at UN summit to discuss climate"],
    ]
)

demo.launch(share=True)  # share=True gives a public URL

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bafa8a60c7686a3576.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
